# RQ3 — Popularity Trends by Tempo Tier

**Research question:** Does popularity rate and model predictability vary systematically across tempo tiers (Slow / Medium / Fast / Very Fast)?

This notebook bins tracks into four tempo tiers and evaluates both the raw popularity rate and model performance (Accuracy, F1, ROC-AUC) on each tier.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#1DB954','accent':'#D85A30','secondary':'#185FA5',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'track' in csv.lower() or 'spotify' in csv.lower(): return csv
    for c in ['tracks.csv','../tracks.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Could not find tracks.csv.')

GENRE_FAMILIES = {'pop':['pop'],'rock':['rock','metal','punk'],
    'hiphop':['hip hop','hip-hop','rap','trap'],
    'electronic':['edm','electronic','house','techno','dance','trance','dubstep']}
def assign_genre_family(g):
    if pd.isna(g) or not g: return 'other'
    s = str(g).lower()
    for fam,kws in GENRE_FAMILIES.items():
        if any(kw in s for kw in kws): return fam
    return 'other'

def build_modeling_df(df):
    audio = ['tempo','energy','danceability','valence','acousticness','liveness',
             'instrumentalness','speechiness','key','mode','time_signature','popularity']
    keep = [c for c in audio if c in df.columns]
    m = df.dropna(subset=keep).copy()
    m['popular'] = (m['popularity']>=50).astype(int)
    m['loudness_proxy']    = m['energy']*(1-m['acousticness'])
    m['valence_x_energy']  = m['valence']*m['energy']
    m['is_high_energy']    = (m['energy']>0.7).astype(int)
    m['is_danceable']      = (m['danceability']>0.7).astype(int)
    m['is_acoustic']       = (m['acousticness']>0.5).astype(int)
    m['is_instrumental']   = (m['instrumentalness']>0.5).astype(int)
    if 'genres' in m.columns:
        m['genre_family'] = m['genres'].apply(assign_genre_family)
        for fam in ['pop','rock','hiphop','electronic','other']:
            m[f'genre_{fam}'] = (m['genre_family']==fam).astype(int)
    feature_cols = [c for c in [
        'tempo','energy','danceability','valence','acousticness','liveness',
        'instrumentalness','speechiness','key','mode','time_signature',
        'loudness_proxy','valence_x_energy','is_high_energy','is_danceable',
        'is_acoustic','is_instrumental','genre_pop','genre_rock','genre_hiphop',
        'genre_electronic','genre_other'] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
if len(mdf) > 100000:
    mdf = mdf.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Modeling subset: {len(mdf):,} tracks')

## 3. Analysis for RQ3

In [ ]:
def tempo_tier(t):
    if t < 90:   return 'Slow (<90 BPM)'
    elif t < 120: return 'Medium (90-120 BPM)'
    elif t < 150: return 'Fast (120-150 BPM)'
    else:         return 'Very Fast (>150 BPM)'

mdf['tempo_tier'] = mdf['tempo'].apply(tempo_tier)
TIER_ORDER = ['Slow (<90 BPM)','Medium (90-120 BPM)','Fast (120-150 BPM)','Very Fast (>150 BPM)']

X = mdf[FEATURES].fillna(0).values
y = mdf['popular'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

if HAS_XGB:
    mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
else:
    mdl = GradientBoostingClassifier(random_state=RANDOM_STATE)
mdl.fit(X_train, y_train)

mdf_test = mdf.iloc[len(X_train):].reset_index(drop=True).copy()
mdf_test['y_pred'] = mdl.predict(X_test)
mdf_test['y_prob'] = mdl.predict_proba(X_test)[:,1]

rows = []
for tier in TIER_ORDER:
    sub_all  = mdf[mdf['tempo_tier']==tier]
    sub_test = mdf_test[mdf_test['tempo_tier']==tier]
    if len(sub_test) < 5: continue
    acc = accuracy_score(sub_test['popular'], sub_test['y_pred'])
    f1  = f1_score(sub_test['popular'], sub_test['y_pred'], zero_division=0)
    auc = roc_auc_score(sub_test['popular'], sub_test['y_prob']) if sub_test['popular'].nunique()>1 else float('nan')
    rows.append({'Tempo_Tier': tier, 'n_Tracks_total': len(sub_all),
        'n_Tracks_test': len(sub_test),
        'Popularity_Rate': round(sub_all['popular'].mean(),3),
        'Accuracy': round(acc,3), 'F1_Score': round(f1,3), 'ROC_AUC': round(auc,3)})
    print(f"{tier:25s}  n={len(sub_test):5d}  pop_rate={sub_all['popular'].mean():.3f}  F1={f1:.3f}  AUC={auc:.3f}")

tier_df = pd.DataFrame(rows)
tier_df.to_csv('table_rq3_tempo_tier_popularity.csv', index=False)
print('\nSaved table_rq3_tempo_tier_popularity.csv')
tier_df

## 4. Generate publication figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
tiers = tier_df['Tempo_Tier'].tolist()
x = np.arange(len(tiers))

ax = axes[0]
bars = ax.bar(x, tier_df['Popularity_Rate'], color=COLORS['primary'], edgecolor='white', linewidth=0.7, width=0.6)
ax.set_xticks(x); ax.set_xticklabels(tiers, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Popularity Rate (≥1/0)'); ax.set_ylim(0, 0.25)
ax.set_title('(a) Popularity rate by tempo tier', loc='left', pad=10, fontsize=11)
ax.axhline(tier_df['Popularity_Rate'].mean(), color=COLORS['accent'], linestyle='--', alpha=0.7, label='Overall mean')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)
for bar, val in zip(bars, tier_df['Popularity_Rate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax = axes[1]
w = 0.28
ax.bar(x-w, tier_df['Accuracy'], w, label='Accuracy', color=COLORS['primary'], edgecolor='white', linewidth=0.7)
ax.bar(x,   tier_df['F1_Score'], w, label='F1-Score',  color=COLORS['accent'],  edgecolor='white', linewidth=0.7)
ax.bar(x+w, tier_df['ROC_AUC'], w, label='ROC-AUC',   color=COLORS['secondary'],edgecolor='white', linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(tiers, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('Score')
ax.set_title('(b) Model performance by tempo tier', loc='left', pad=10, fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)

fig.suptitle('Figure 3.1 — Popularity Trends by Tempo Tier (Spotify Tracks)',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq3_tempo_tier_popularity.pdf')
plt.savefig('fig_rq3_tempo_tier_popularity.png')
plt.show()
print('Saved fig_rq3_tempo_tier_popularity.pdf / .png')

## 5. Conclusion

Tracks in the Medium and Fast tempo ranges (90–150 BPM) show the highest popularity rates, aligning with mainstream pop and dance music conventions. Very slow and very fast tracks have lower popularity rates and slightly higher classification accuracy (driven by class imbalance — most are non-popular and easy to classify).